## Author**Kholipha Ahmmad Al-Amin**Software Engineer, SaaS Founder, and AI/Data Practitioner from Dhaka, Bangladesh.Portfolio: https://kholipha-ahmmad-al-amin.equisaas-bd.comGitHub: https://github.com/kholipha-ahmmad-al-aminLinkedIn: https://www.linkedin.com/in/kholipha-ahmmad-al-amin

# **Final Year Project**

## **Predictive Analytics for Electricity Loadshedding in Bangladesh Grid (2025 Dataset)**

**Student Name:** Azmira Akter  
**Student ID:** 222-0253-203  
**Supervisor:** Md Ohidul Islam, Assistant Professor, Department of CSE  
**Department of Computer Science and Engineering**  
**Atish Dipankar University of Science and Technology**  

---

**Project Overview**  
This notebook develops an advanced machine learning system to predict electricity loadshedding in Bangladesh using the 2025 Grid Dataset. The project combines time-series forecasting, classification of loadshedding events, and energy source contribution analysis to provide actionable insights for grid stability and renewable integration.

**Key Objectives:**  
- Predict hourly loadshedding amount and occurrence  
- Analyze contribution of different energy sources (gas, solar, hydro, etc.)  
- Identify peak demand patterns and critical periods  
- Deploy an interactive dashboard for real-time scenario analysis

**Dataset Source**  
Bangladesh Grid Electricity Loadshedding Dataset (2025) – UCI Machine Learning Repository  
Link: https://archive.ics.uci.edu/dataset/1045/bangladesh+grid+electricity+loadshedding+dataset+2025

**Date:** January 03, 2026

## **Notebook Structure**

1. Environment Setup  
2. Project Organization  
3. Dataset Loading & Initial Inspection  
4. Comprehensive Exploratory Data Analysis  
5. Feature Engineering & Preprocessing  
6. Model Development (Forecasting + Classification)  
7. Training & Evaluation  
8. Performance Analysis & Insights  
9. Model Persistence  
10. Premium Interactive Gradio Dashboard

In [ ]:
# @title 1. Install Required Packages
%%capture
!pip install --quiet --upgrade pip
!pip install --quiet pandas numpy matplotlib seaborn plotly scikit-learn xgboost lightgbm gradio

In [ ]:
# @title 2. Imports & Configuration
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

# Premium Styling
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

In [ ]:
# @title 3. Project Organization
from google.colab import drive
drive.mount('/content/drive')

PROJECT_NAME = 'Bangladesh_Grid_Loadshedding_Project'
BASE_DIR = Path('/content/drive/MyDrive') / PROJECT_NAME

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = BASE_DIR / f'run_{RUN_ID}'
FIG_DIR = RUN_DIR / 'figures'
MODEL_DIR = RUN_DIR / 'models'

for folder in [BASE_DIR, RUN_DIR, FIG_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project outputs saved to: {RUN_DIR}")

In [ ]:
# @title 4. Dataset Loading
# Upload the CSV file from UCI to your Drive, or download directly

DATA_PATH = Path('/content/drive/MyDrive/bangladesh_grid_loadshedding_2025.csv')  # Update with your file name

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}. Please upload the CSV.")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

## **5. Exploratory Data Analysis**

In [ ]:
# @title 5.1 Dataset Overview & Summary Statistics
print("Dataset Info:")
df.info()

print("\nSummary Statistics:")
df.describe()

In [ ]:
# @title 5.2 Time Series Visualization - Demand vs Generation
# Assuming columns: 'DateTime', 'Demand_MW', 'Generation_MW', 'Loadshedding_MW'

df['DateTime'] = pd.to_datetime(df['DateTime'])  # Adjust column name if different
df = df.sort_values('DateTime')

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['DateTime'], y=df['Demand_MW'], name='Demand (MW)'))
fig.add_trace(go.Scatter(x=df['DateTime'], y=df['Generation_MW'], name='Generation (MW)'))
fig.add_trace(go.Scatter(x=df['DateTime'], y=df['Loadshedding_MW'], name='Loadshedding (MW)', fill='tonexty'))
fig.update_layout(title='Bangladesh Grid: Demand, Generation & Loadshedding Over Time', height=600)
fig.show()

plt.savefig(FIG_DIR / 'demand_generation_loadshedding.png', dpi=300)

In [ ]:
# @title 5.3 Energy Source Contribution (Stacked Area Chart)
# Adjust column names based on actual dataset (e.g., Gas_MW, Solar_MW, etc.)
source_cols = [col for col in df.columns if col.endswith('_MW') and col not in ['Demand_MW', 'Generation_MW', 'Loadshedding_MW']]

fig = go.Figure()
for col in source_cols:
    fig.add_trace(go.Scatter(x=df['DateTime'], y=df[col], name=col.replace('_MW', ''), stackgroup='one'))
fig.update_layout(title='Energy Mix Contribution Over Time', height=600)
fig.show()

plt.savefig(FIG_DIR / 'energy_source_contribution.png', dpi=300)

In [ ]:
# @title 5.4 Peak Demand Hours Analysis
df['Hour'] = df['DateTime'].dt.hour
hourly_avg = df.groupby('Hour')[['Demand_MW', 'Loadshedding_MW']].mean()

fig = px.line(hourly_avg, title='Average Demand & Loadshedding by Hour of Day')
fig.show()

plt.savefig(FIG_DIR / 'peak_hours.png', dpi=300)

## **6. Feature Engineering & Preprocessing**

In [ ]:
# @title 6.1 Create Time-Based Features
df['Month'] = df['DateTime'].dt.month
df['DayOfWeek'] = df['DateTime'].dt.dayofweek
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
df['PeakHour'] = df['Hour'].apply(lambda x: 1 if 18 <= x <= 22 else 0)

# Lag features (previous hour demand)
df['Demand_Lag1'] = df['Demand_MW'].shift(1)
df['Loadshedding_Lag1'] = df['Loadshedding_MW'].shift(1)

# Rolling statistics
df['Demand_RollingMean_6h'] = df['Demand_MW'].rolling(window=6).mean()

df = df.dropna()  # Remove rows with NaN from lagging

In [ ]:
# @title 6.2 Prepare Features & Targets
feature_cols = ['Month', 'DayOfWeek', 'IsWeekend', 'PeakHour', 'Hour',
                'Demand_Lag1', 'Demand_RollingMean_6h'] + source_cols

X = df[feature_cols]
y_regression = df['Loadshedding_MW']  # Predict amount
y_classification = (df['Loadshedding_MW'] > 0).astype(int)  # Binary: shedding or not

X_train, X_test, y_reg_train, y_reg_test = train_test_split(X, y_regression, test_size=0.2, shuffle=False)
_, _, y_cls_train, y_cls_test = train_test_split(X, y_classification, test_size=0.2, shuffle=False)

## **7. Model Development**

In [ ]:
# @title 7.1 XGBoost Regressor for Loadshedding Amount
reg_model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=8, random_state=42)
reg_model.fit(X_train, y_reg_train)

reg_pred = reg_model.predict(X_test)
print(f"Regression MAE: {mean_absolute_error(y_reg_test, reg_pred):.2f} MW")
print(f"Regression R²: {r2_score(y_reg_test, reg_pred):.4f}")

In [ ]:
# @title 7.2 LightGBM Classifier for Loadshedding Occurrence
cls_model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, max_depth=10, random_state=42)
cls_model.fit(X_train, y_cls_train)

cls_pred = cls_model.predict(X_test)
print("Classification Report:")
print(classification_report(y_cls_test, cls_pred))

## **8. Performance Visualization**

In [ ]:
# @title 8.1 Actual vs Predicted Loadshedding
test_df = df.iloc[-len(X_test):].copy()
test_df['Predicted_Loadshedding'] = reg_pred

fig = go.Figure()
fig.add_trace(go.Scatter(x=test_df['DateTime'], y=test_df['Loadshedding_MW'], name='Actual'))
fig.add_trace(go.Scatter(x=test_df['DateTime'], y=test_df['Predicted_Loadshedding'], name='Predicted'))
fig.update_layout(title='Actual vs Predicted Loadshedding (Test Period)', height=600)
fig.show()

plt.savefig(FIG_DIR / 'prediction_comparison.png', dpi=300)

## **9. Model Persistence**

In [ ]:
# @title 9.1 Save Models
import joblib
joblib.dump(reg_model, MODEL_DIR / 'loadshedding_regressor.pkl')
joblib.dump(cls_model, MODEL_DIR / 'loadshedding_classifier.pkl')

metadata = {
    'student_name': 'Azmira Akter',
    'student_id': '222-0253-203',
    'supervisor': 'Md Ohidul Islam',
    'project_title': 'Predictive Analytics for Electricity Loadshedding in Bangladesh',
    'dataset': 'Bangladesh Grid Electricity Loadshedding Dataset (2025)',
    'run_date': RUN_ID
}

with open(MODEL_DIR / 'project_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Models and metadata saved.")

## **10. Premium Interactive Gradio Dashboard**

In [ ]:
# @title ⚡ Bangladesh Grid Loadshedding Predictor – Premium Gradio Dashboard

!pip install --quiet gradio plotly pandas

import gradio as gr
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import joblib
from datetime import datetime, timedelta

# Load models
reg_model = joblib.load(MODEL_DIR / 'loadshedding_regressor.pkl')
cls_model = joblib.load(MODEL_DIR / 'loadshedding_classifier.pkl')

# Sample recent data for context
recent_data = df.tail(24)  # Last 24 hours

def predict_loadshedding(date, hour, demand_mw, gas_mw, solar_mw, hydro_mw, coal_mw, oil_mw):
    # Create feature vector
    dt = datetime.strptime(f"{date} {hour}:00", "%Y-%m-%d %H")
    features = {
        'Month': dt.month,
        'DayOfWeek': dt.weekday(),
        'IsWeekend': 1 if dt.weekday() >= 5 else 0,
        'PeakHour': 1 if 18 <= dt.hour <= 22 else 0,
        'Hour': dt.hour,
        'Demand_Lag1': demand_mw,  # Simplified
        'Demand_RollingMean_6h': demand_mw,
        'Gas_MW': gas_mw,
        'Solar_MW': solar_mw,
        'Hydro_MW': hydro_mw,
        'Coal_MW': coal_mw,
        'Oil_MW': oil_mw
    }
    
    X = pd.DataFrame([features])
    
    pred_amount = reg_model.predict(X)[0]
    pred_prob = cls_model.predict_proba(X)[0][1]
    risk_level = "High" if pred_prob > 0.7 else "Medium" if pred_prob > 0.4 else "Low"
    
    # Visualization
    fig = go.Figure(go.Indicator(
        mode = "gauge+number+delta",
        value = pred_amount,
        domain = {'x': [0, 1], 'y': [0, 1]},
        title = {'text': "Predicted Loadshedding (MW)"},
        gauge = {
            'axis': {'range': [0, max(500, pred_amount * 1.2)]},
            'bar': {'color': "darkblue"},
            'steps': [
                {'range': [0, 100], 'color': "lightgreen"},
                {'range': [100, 300], 'color': "yellow"},
                {'range': [300, 500], 'color': "red"}
            ],
            'threshold': {'value': pred_amount}
        }
    ))
    
    result_html = f"""
    <div style="padding:30px; background:white; border-radius:20px; box-shadow:0 10px 30px rgba(0,0,0,0.1); text-align:center;">
        <h2 style="color:#dc2626;">Risk Level: {risk_level}</h2>
        <p><strong>Predicted Loadshedding:</strong> {pred_amount:.1f} MW</p>
        <p><strong>Probability of Loadshedding:</strong> {pred_prob:.1%}</p>
    </div>
    """
    
    return result_html, fig

theme = gr.themes.Soft(primary_hue="red", font=[gr.themes.GoogleFont("Poppins")])

with gr.Blocks(theme=theme, title="Bangladesh Grid Loadshedding Predictor") as demo:
    gr.HTML("""
    <div style="text-align:center; padding:40px; background:linear-gradient(135deg,#7f1d1d,#dc2626); color:white; border-radius:20px;">
        <h1>Bangladesh Grid Loadshedding Prediction System</h1>
        <p>Final Year Project • Azmira Akter (222-0253-203) • Supervisor: Md Ohidul Islam</p>
    </div>
    """)
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Input Scenario")
            date = gr.Date(label="Date")
            hour = gr.Slider(0, 23, step=1, label="Hour of Day")
            demand_mw = gr.Number(label="Expected Demand (MW)", value=8000)
            gas_mw = gr.Number(label="Gas Generation (MW)", value=5000)
            solar_mw = gr.Number(label="Solar Generation (MW)", value=300)
            hydro_mw = gr.Number(label="Hydro Generation (MW)", value=800)
            coal_mw = gr.Number(label="Coal Generation (MW)", value=1000)
            oil_mw = gr.Number(label="Oil Generation (MW)", value=500)
            
            predict_btn = gr.Button("Predict Loadshedding Risk", variant="primary")
        
        with gr.Column():
            gr.Markdown("### Prediction Result")
            output_html = gr.HTML()
            output_gauge = gr.Plot()
    
    predict_btn.click(
        predict_loadshedding,
        inputs=[date, hour, demand_mw, gas_mw, solar_mw, hydro_mw, coal_mw, oil_mw],
        outputs=[output_html, output_gauge]
    )

demo.launch(share=True)